# V12 Crisis Mining (Colab L4/T4)

Bulk of the V12 training corpus comes from crisis. Each replay starts from a near-death snapshot of a policy-only probe and plays forward `--continue-turns 500` turns of strong-search MCTS — capturing both the recovery transition AND post-recovery steady-state play.

Player: **pillar2y2_epoch_40 + value_head_v11 + q_weight=2.0** (Phase 3R: mean 13,476).

Settings (per V12 design):
- recovery_turns 15 / prevention_turns 35 (snapshot offsets from death)
- recovery_sims 600 / prevention_sims 400
- continue_turns 500 (strong-search post-snapshot play)
- policy-probe cap 5000 turns (matches deployment)
- batch_size 8 (HISTORY 115)

**Upload to `MyDrive/alphatrain/` before running:**
1. `colorlines_selfplay_train.tar.gz` (built by `scripts/build_selfplay_tarball.sh`)
2. `pillar2y2_epoch_40.pt` (~143 MB)
3. `value_head_v11.pt` (~38 KB)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/alphatrain'

!cp {DRIVE}/colorlines_selfplay_train.tar.gz /content/
!cd /content && tar xzf colorlines_selfplay_train.tar.gz

os.makedirs('/content/alphatrain/data', exist_ok=True)
!cp {DRIVE}/pillar2y2_epoch_40.pt /content/alphatrain/data/pillar2y2_epoch_40.pt
!cp {DRIVE}/value_head_v11.pt /content/alphatrain/data/value_head_v11.pt
print(f'Model: {os.path.getsize("/content/alphatrain/data/pillar2y2_epoch_40.pt")/1e6:.0f} MB')
print(f'Value head: {os.path.getsize("/content/alphatrain/data/value_head_v11.pt")/1e3:.0f} KB')

!pip install -q numpy numba scipy

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {g.total_memory / 1e9:.1f} GB')

In [ ]:
# === V12 Crisis Mining ===
# 20K total seeds split across instances if multiple Colab sessions:
#   Instance 1: 1300000-1305000
#   Instance 2: 1305000-1310000
#   Instance 3: 1310000-1315000
#   Instance 4: 1315000-1320000
SEED_START = 1300000
SEED_END = 1305000     # 5K seeds per instance — fits ~6-10h on L4
# ============================

Q_WEIGHT = 2.0
RECOVERY_TURNS = 15
PREVENTION_TURNS = 35
RECOVERY_SIMS = 600
PREVENTION_SIMS = 400
CONTINUE_TURNS = 500   # post-snapshot play length — equilibrates back to steady-state
POLICY_MAX_TURNS = 5000
WORKERS = 20
BATCH_SIZE = 8         # HISTORY 115
SAVE_DIR = f'{DRIVE}/crisis_v12'

os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Seeds: {SEED_START}-{SEED_END-1} ({SEED_END-SEED_START} probes)')
print(f'q_weight: {Q_WEIGHT}, recovery {RECOVERY_TURNS}t @ {RECOVERY_SIMS} sims, prevention {PREVENTION_TURNS}t @ {PREVENTION_SIMS} sims')
print(f'Continue: {CONTINUE_TURNS} turns post-snapshot')
print(f'Policy probe cap: {POLICY_MAX_TURNS}, workers: {WORKERS}, bs: {BATCH_SIZE}')
print(f'Save: {SAVE_DIR}')

!python -m alphatrain.scripts.crisis_mining \
    --model /content/alphatrain/data/pillar2y2_epoch_40.pt \
    --value-head-path /content/alphatrain/data/value_head_v11.pt \
    --q-weight {Q_WEIGHT} \
    --seed-start {SEED_START} --seed-end {SEED_END} \
    --recovery-turns {RECOVERY_TURNS} --recovery-sims {RECOVERY_SIMS} \
    --prevention-turns {PREVENTION_TURNS} --prevention-sims {PREVENTION_SIMS} \
    --continue-turns {CONTINUE_TURNS} \
    --policy-max-turns {POLICY_MAX_TURNS} \
    --max-turns 5000 \
    --workers {WORKERS} --device cuda --batch-size {BATCH_SIZE} \
    --compile \
    --save-dir {SAVE_DIR}

## Notes

- **Two-phase pipeline:** Phase 1 = policy-only probes (fast, ~few minutes for 5K seeds). Phase 2 = MCTS replays (dominates wall time).
- **Replay yield:** strong policy mostly survives the 5000-turn probe cap, so per-seed snapshot yield is lower than V11. May want to over-provision seed count if too few replays land. The output dir tells you total replay count.
- **Resume support:** re-running this cell skips already-completed `(seed, label)` pairs, so disconnects are safe.
- **Wall time on L4:** ~6-10h for 5K seeds at 600/400 sims with 500 continue-turns. T4 ~1.5× slower. Run multiple instances in parallel for the full 20K.
- **Watch:** the `[GPU] X evals, Y fwd (avg bs=Z, ZZZZ evals/s)` line. avg bs ≥50 and evals/s in low thousands means GPU is saturated; if either drops, check that policy probes aren't all hitting the cap (in which case Phase 2 starves).
- **Output structure:** `game_seed{N}_recovery_score{S}.json`, `game_seed{N}_prevention_score{S}.json`. Build script (run on M5) merges with self-play games into the V12 training tensor.